## alpha_tutorial — AlphaEarth K-Means Clustering Intro

**Purpose**: Introduction to AlphaEarth 64-dim satellite embeddings via K-means clustering.

**Inputs**: Google Earth Engine — `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`
**Outputs**: Interactive geemap visualization in-notebook (no files written)
**GEE auth required**: Yes

**Parameters**:
- Number of clusters: 3, 5, or 10
- Region: configurable AOI

**Expected runtime**: < 2 min (server-side EE computation)

**How to run**: Execute cells top-to-bottom after GEE authentication.


# AlphaEarth Embeddings Python Tutorial

## Clustering Example

This python was converted from the original javascript here:

https://developers.google.com/earth-engine/tutorials/community/satellite-embedding-01-introduction

In [1]:
import ee
import geemap

# ee.Authenticate())
ee.Initialize(project='ardent-fusion-421917')

Use the satellite basemap (Note: Map.setOptions is specific to Code Editor)

In Python, you'll typically use geemap or folium for visualization

In [2]:
embeddings = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

geometry = ee.Geometry.Polygon(
    [[[76.3978, 12.5521], [76.3978, 12.3550], [76.6519, 12.3550], [76.6519, 12.5521]]]
)

In [3]:
year = 2024
start_date = ee.Date.fromYMD(year, 1, 1)
end_date = start_date.advance(1, "year")

filtered_embeddings = embeddings.filter(ee.Filter.date(start_date, end_date)).filter(
    ee.Filter.bounds(geometry)
)

In [4]:
embeddings_image = filtered_embeddings.mosaic()
print("Satellite Embedding Image", embeddings_image.getInfo())

Satellite Embedding Image {'type': 'Image', 'bands': [{'id': 'A00', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A01', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A02', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A03', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A04', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A05', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A06', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A07', 'data_type': {'

In [5]:
n_samples = 1000
training = embeddings_image.sample(
    region=geometry, scale=10, numPixels=n_samples, seed=100
)
print(training.first().getInfo())

{'type': 'Feature', 'geometry': None, 'id': '0', 'properties': {'A00': 0.08421376393694734, 'A01': -0.07111111111111111, 'A02': 0.13588619761630144, 'A03': 0.010396001537870049, 'A04': -0.0271280276816609, 'A05': -0.04822760476739715, 'A06': 0.008858131487889272, 'A07': 0.12456747404844293, 'A08': -0.12456747404844293, 'A09': -0.03844675124951941, 'A10': 0.024605920799692427, 'A11': -0.008858131487889272, 'A12': -0.3839138792772011, 'A13': 0.11909265667051133, 'A14': -0.1085121107266436, 'A15': -0.07111111111111111, 'A16': 0.019930795847750864, 'A17': 0.10340638216070744, 'A18': -0.11909265667051133, 'A19': 0.11909265667051133, 'A20': -0.24415224913494812, 'A21': 0.2761399461745483, 'A22': 0.24415224913494812, 'A23': -0.0271280276816609, 'A24': -0.07535563244905807, 'A25': -0.019930795847750864, 'A26': 0.05173394848135333, 'A27': -0.04822760476739715, 'A28': -0.07972318339100345, 'A29': 0.11909265667051133, 'A30': 0.022206843521722412, 'A31': 0.024605920799692427, 'A32': -0.03844675124

In [6]:
# Function to train a model for desired number of clusters
def get_clusters(n_clusters):
    clusterer = ee.Clusterer.wekaKMeans(n_clusters).train(training)

    # Cluster the image
    clustered = embeddings_image.cluster(clusterer)
    return clustered


cluster3 = get_clusters(3)

cluster5 = get_clusters(5)

cluster10 = get_clusters(10)

To visualize the results in Python, you can use geemap:

In [7]:
vis_params = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}

Map = geemap.Map()
Map.centerObject(geometry, 12)
Map.addLayer(embeddings_image.clip(geometry), vis_params, 'Embeddings Image')
Map.addLayer(cluster3.randomVisualizer().clip(geometry), {}, '3 clusters')
Map.addLayer(cluster5.randomVisualizer().clip(geometry), {}, '5 clusters')
Map.addLayer(cluster10.randomVisualizer().clip(geometry), {}, '10 clusters')
Map

Map(center=[12.453567183649733, 76.52485000000064], controls=(WidgetControl(options=['position', 'transparent_…